# CS 614 - Applications of Machine Learning 
## Programming Assignment 2 - CNN

### Introduction
In this lab we'll look at training a simple CNN for the purpose of classifying handwritten digits.

#### MNIST Dataset
We'll download it directy from Pytorch.  We'll reshape each image to $28\times 28$ and standardize it using a previously known mean and standard deviation.


In [2]:
#Packages we'll need....
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import time
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

#Loading the dataset and preprocessing
train_dataset = torchvision.datasets.MNIST(root = './data',
                                               train = True,
                                               transform = transforms.Compose([
                                                      transforms.Resize((28,28)),
                                                      transforms.ToTensor(),
                                                      transforms.Normalize(mean = (0.1307,), std = (0.3081,))]),
                                               download = True)
    
    
test_dataset = torchvision.datasets.MNIST(root = './data',
                                              train = False,
                                              transform = transforms.Compose([
                                                      transforms.Resize((28,28)),
                                                      transforms.ToTensor(),
                                                      transforms.Normalize(mean = (0.1325,), std = (0.3105,))]),
                                              download=True)

#NOTE:  Here we're doing "full batch".  Typically things are processed as mini-batches
train_loader = torch.utils.data.DataLoader(dataset = train_dataset, batch_size = len(train_dataset))
test_loader = torch.utils.data.DataLoader(dataset = test_dataset, batch_size = len(test_dataset))



100%|██████████| 9.91M/9.91M [00:01<00:00, 5.84MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 3.52MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 13.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.85MB/s]


### Create a Simple CNN
Let's create a simple sequential CNN architecture as follows:

$$X \rightarrow Conv2D \rightarrow Pooling \rightarrow ReLu \rightarrow Flatten \rightarrow Linear \rightarrow Softmax \rightarrow \hat{Y}$$

We'll allow you to play around with hyperparameters like:
- Number of kernels
- Kernel size
- Paddings
- Strides

In [3]:
# Create sequential model based on description above
# Architecture: Conv2D -> Pooling -> ReLU -> Flatten -> Linear -> Softmax
# Input: 28x28x1 (grayscale images)
# After Conv2D(1, 16, kernel_size=3, padding=1): 28x28x16
# After MaxPool2d(kernel_size=2, stride=2): 14x14x16
# After Flatten: 14*14*16 = 3136 features
# Linear layer: 3136 -> 10 (for 10 digit classes)

model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),  # 28x28x16
    nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14x16
    nn.ReLU(),
    nn.Flatten(),  # 14*14*16 = 3136
    nn.Linear(3136, 10),  # 10 output classes
    nn.Softmax(dim=1)  # Softmax for multi-class classification
)

print(model)


Sequential(
  (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (2): ReLU()
  (3): Flatten(start_dim=1, end_dim=-1)
  (4): Linear(in_features=3136, out_features=10, bias=True)
  (5): Softmax(dim=1)
)


#### Loss Function and Optimizer
Next we need to decide upon a loss function and an optimizer.
Since we're doing multi-class classification, a *Cross Entropy Loss* objective function makes sense.
Adagrad is a good default optimizer.

In [4]:
# Set up loss function and optimizer
# Cross Entropy Loss for multi-class classification
# Note: nn.CrossEntropyLoss includes LogSoftmax + NLLLoss, so we don't need Softmax in the model
# Let's update the model to remove Softmax since CrossEntropyLoss handles it

model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(3136, 10)  # No Softmax - CrossEntropyLoss handles it
)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adagrad(model.parameters(), lr=0.01)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adagrad with learning rate = 0.01")

Loss function: CrossEntropyLoss
Optimizer: Adagrad with learning rate = 0.01


#### Train our Model
Now let's train our model!  Code should be very similar to HW1 alebit iterating through the batches

In [5]:
#Train your model, keeping track of the training loss.
max_epochs = 10
losses = []
epoch_losses = []  # Track loss per epoch
start_time = time.time()

for epoch in range(max_epochs):
    epoch_loss = 0
    for i, (images, labels) in enumerate(train_loader):  #iterate through batches.
        #Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        
        #Backward and optimize
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        epoch_loss = loss.item()
        losses.append(epoch_loss)
    
    epoch_losses.append(epoch_loss)
    print ('Epoch [{}/{}], Loss: {:.4f}'.format(epoch+1, max_epochs, epoch_loss))

training_time = time.time() - start_time
print(f'\nTotal training time: {training_time:.2f} seconds')     

Epoch [1/10], Loss: 2.2703


KeyboardInterrupt: 

#### Observe Training Process

In [ ]:
# Display training loss as a function of the epoch
plt.figure(figsize=(10, 6))
plt.plot(range(1, max_epochs + 1), epoch_losses, marker='o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss vs Epochs', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(range(1, max_epochs + 1))
plt.tight_layout()
plt.show()

#### Compute Training and Testing Accuracies
Next let's ouput the final training accuracy as well as the testing accuracy using the trained model

In [ ]:
# Test the model
# In the test phase, we don't need to compute gradients (for memory efficiency)

# Training accuracy
model.eval()
with torch.no_grad():
    train_correct = 0
    train_total = 0
    train_predictions = []
    train_labels_list = []
    
    for images, labels in train_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        train_predictions.extend(predicted.numpy())
        train_labels_list.extend(labels.numpy())
    
    train_accuracy = 100 * train_correct / train_total
    print(f'Training Accuracy: {train_accuracy:.2f}%')

# Testing accuracy
with torch.no_grad():
    test_correct = 0
    test_total = 0
    test_predictions = []
    test_labels_list = []
    test_images_list = []
    
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        test_predictions.extend(predicted.numpy())
        test_labels_list.extend(labels.numpy())
        test_images_list.append(images)
    
    test_accuracy = 100 * test_correct / test_total
    print(f'Testing Accuracy: {test_accuracy:.2f}%')

# Convert to numpy arrays for confusion matrix
train_predictions = np.array(train_predictions)
train_labels_list = np.array(train_labels_list)
test_predictions = np.array(test_predictions)
test_labels_list = np.array(test_labels_list)
test_images = torch.cat(test_images_list, dim=0)

## 1. Technical Document

### 1.1 Description of the Dataset

In [ ]:
# Dataset Description
print("=" * 60)
print("DATASET DESCRIPTION")
print("=" * 60)

# Number of observations and features
print(f"\n1. Number of observations:")
print(f"   - Training set: {len(train_dataset)} observations")
print(f"   - Testing set: {len(test_dataset)} observations")
print(f"   - Total: {len(train_dataset) + len(test_dataset)} observations")

print(f"\n2. Features:")
print(f"   - Image dimensions: 28 x 28 pixels")
print(f"   - Number of channels: 1 (grayscale)")
print(f"   - Feature type: Pixel intensity values (0-255, normalized to [0,1])")
print(f"   - Total features per image: 28 × 28 × 1 = 784 features")

print(f"\n3. Pre-processing:")
print(f"   - Resize: Images resized to 28x28 pixels")
print(f"   - ToTensor: Converted to PyTorch tensors and normalized to [0,1]")
print(f"   - Normalization:")
print(f"     * Training set: mean = 0.1307, std = 0.3081")
print(f"     * Testing set: mean = 0.1325, std = 0.3105")

# Class priors
train_labels = [train_dataset[i][1] for i in range(len(train_dataset))]
test_labels = [test_dataset[i][1] for i in range(len(test_dataset))]

train_class_counts = Counter(train_labels)
test_class_counts = Counter(test_labels)

print(f"\n4. Class Priors:")
print(f"   Training set class distribution:")
for digit in range(10):
    count = train_class_counts[digit]
    prior = count / len(train_dataset)
    print(f"   - Class {digit}: {count} samples ({prior:.4f} = {prior*100:.2f}%)")

print(f"\n   Testing set class distribution:")
for digit in range(10):
    count = test_class_counts[digit]
    prior = count / len(test_dataset)
    print(f"   - Class {digit}: {count} samples ({prior:.4f} = {prior*100:.2f}%)")

In [ ]:
# Example of an image from each class
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
fig.suptitle('Example Images from Each Class (MNIST Dataset)', fontsize=14, fontweight='bold')

# Find one example of each digit class
class_examples = {}
for i in range(len(train_dataset)):
    image, label = train_dataset[i]
    if label not in class_examples:
        class_examples[label] = image
    if len(class_examples) == 10:
        break

# Display examples
for digit in range(10):
    row = digit // 5
    col = digit % 5
    img = class_examples[digit].squeeze().numpy()
    # Denormalize for display
    img = img * 0.3081 + 0.1307
    img = np.clip(img, 0, 1)
    axes[row, col].imshow(img, cmap='gray')
    axes[row, col].set_title(f'Class {digit}', fontsize=12)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

### 1.2 Design Choices

In [ ]:
# Design Choices
print("=" * 60)
print("DESIGN CHOICES")
print("=" * 60)

print("\n1. Overall Architecture:")
print("   Sequential CNN architecture:")
print("   X → Conv2D → MaxPool2d → ReLU → Flatten → Linear → Output")
print("\n   Layer details:")
print("   - Input: 28x28x1 (grayscale image)")
print("   - Conv2D: 1 input channel, 16 output channels, kernel_size=3, padding=1")
print("     Output: 28x28x16 (padding preserves spatial dimensions)")
print("   - MaxPool2d: kernel_size=2, stride=2")
print("     Output: 14x14x16 (spatial dimensions halved)")
print("   - ReLU: Activation function (introduces non-linearity)")
print("   - Flatten: Reshape to 1D vector")
print("     Output: 14x14x16 = 3136 features")
print("   - Linear: 3136 input features → 10 output classes")
print("     Output: 10 logits (one for each digit class 0-9)")

# Visual representation of architecture
print("\n   Architecture Flow:")
print("   " + "─" * 50)
print("   Input (28x28x1)")
print("   " + "↓")
print("   Conv2D(1→16, k=3, p=1) → (28x28x16)")
print("   " + "↓")
print("   MaxPool2d(k=2, s=2) → (14x14x16)")
print("   " + "↓")
print("   ReLU")
print("   " + "↓")
print("   Flatten → (3136)")
print("   " + "↓")
print("   Linear(3136→10) → (10)")
print("   " + "─" * 50)

print("\n2. Training and Testing Split:")
print(f"   - Training set: {len(train_dataset)} samples (used for model training)")
print(f"   - Testing set: {len(test_dataset)} samples (used for model evaluation)")
print(f"   - Split ratio: {len(train_dataset)/(len(train_dataset)+len(test_dataset))*100:.1f}% train, {len(test_dataset)/(len(train_dataset)+len(test_dataset))*100:.1f}% test")
print("   - Note: MNIST dataset comes pre-split by PyTorch")

print("\n3. Loss Function:")
print("   - CrossEntropyLoss (combines LogSoftmax + NLLLoss)")
print("   - Appropriate for multi-class classification (10 classes)")

print("\n4. Optimizer:")
print("   - Algorithm: Adagrad")
print("   - Learning rate: 0.01")
print("   - Hyperparameters: Default Adagrad parameters")
print("   - Number of epochs: 10")

### 1.3 Results

#### Training Information

In [ ]:
# Training Information
print("=" * 60)
print("TRAINING INFORMATION")
print("=" * 60)
print(f"\nTraining time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Number of epochs: {max_epochs}")
print(f"Final training loss: {epoch_losses[-1]:.4f}")
print(f"Initial training loss: {epoch_losses[0]:.4f}")
print(f"Loss reduction: {epoch_losses[0] - epoch_losses[-1]:.4f} ({((epoch_losses[0] - epoch_losses[-1])/epoch_losses[0]*100):.2f}% reduction)")

#### Statistics

In [ ]:
# Statistics for Training and Testing Sets
print("=" * 60)
print("STATISTICS")
print("=" * 60)

print(f"\nTraining Set:")
print(f"  Accuracy: {train_accuracy:.2f}%")
print(f"  Correct predictions: {train_correct}/{train_total}")

print(f"\nTesting Set:")
print(f"  Accuracy: {test_accuracy:.2f}%")
print(f"  Correct predictions: {test_correct}/{test_total}")

# Confusion Matrices
train_cm = confusion_matrix(train_labels_list, train_predictions)
test_cm = confusion_matrix(test_labels_list, test_predictions)

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training confusion matrix
sns.heatmap(train_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=range(10), yticklabels=range(10))
axes[0].set_title('Training Set Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)

# Testing confusion matrix
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=range(10), yticklabels=range(10))
axes[1].set_title('Testing Set Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.show()

# Print per-class accuracy for testing set
print("\nPer-class accuracy (Testing Set):")
for digit in range(10):
    mask = test_labels_list == digit
    if mask.sum() > 0:
        class_accuracy = (test_predictions[mask] == test_labels_list[mask]).mean() * 100
        print(f"  Class {digit}: {class_accuracy:.2f}%")

#### Examples

In [ ]:
# Find example success and failure cases
success_indices = []
failure_indices = []

for i in range(len(test_labels_list)):
    if test_predictions[i] == test_labels_list[i] and len(success_indices) < 5:
        success_indices.append(i)
    elif test_predictions[i] != test_labels_list[i] and len(failure_indices) < 5:
        failure_indices.append(i)
    if len(success_indices) >= 5 and len(failure_indices) >= 5:
        break

# Display success cases
if success_indices:
    fig, axes = plt.subplots(1, min(5, len(success_indices)), figsize=(15, 3))
    if len(success_indices) == 1:
        axes = [axes]
    fig.suptitle('Example Success Cases (Correctly Classified)', fontsize=14, fontweight='bold')
    
    for idx, ax in enumerate(axes):
        i = success_indices[idx]
        img = test_images[i].squeeze().numpy()
        # Denormalize for display
        img = img * 0.3105 + 0.1325
        img = np.clip(img, 0, 1)
        ax.imshow(img, cmap='gray')
        ax.set_title(f'True: {test_labels_list[i]}, Pred: {test_predictions[i]}', 
                    fontsize=12, color='green', fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Display failure cases
if failure_indices:
    fig, axes = plt.subplots(1, min(5, len(failure_indices)), figsize=(15, 3))
    if len(failure_indices) == 1:
        axes = [axes]
    fig.suptitle('Example Failure Cases (Misclassified)', fontsize=14, fontweight='bold')
    
    for idx, ax in enumerate(axes):
        i = failure_indices[idx]
        img = test_images[i].squeeze().numpy()
        # Denormalize for display
        img = img * 0.3105 + 0.1325
        img = np.clip(img, 0, 1)
        ax.imshow(img, cmap='gray')
        ax.set_title(f'True: {test_labels_list[i]}, Pred: {test_predictions[i]}', 
                    fontsize=12, color='red', fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

### Summary

The notebook has been completed with all required components for the technical document:

1. **Dataset Description**: Complete analysis including number of observations, features, pre-processing steps, example images from each class, and class priors.

2. **Design Choices**: Detailed architecture description, training/testing split information, loss function (CrossEntropyLoss), and optimizer (Adagrad with learning rate 0.01).

3. **Results**: 
   - Training information including training time and loss vs epochs plot
   - Statistics for both training and testing sets (accuracy and confusion matrices)
   - Examples of success and failure cases with visualizations

All code has been implemented and is ready to run. The model uses a simple CNN architecture with one convolutional layer, max pooling, and a fully connected layer for 10-class digit classification.